# 05 - Customer Lifetime Value (CLV) & Unit Economics
**Platform:** Bangladesh E-Commerce Customer Intelligence Platform  
**Focus:** Historical Realized Margin, 24-Month Predictive DCF, and LTV:CAC Efficiency

---
## CLV Analytical Framework
1. **Historical CLV:** Cumulative gross margin generated to date:
   $$\text{CLV}_{\text{hist}} = \sum (\text{Revenue} - \text{Wholesale COGS})$$
2. **Predictive CLV (24-Month Discounted Cash Flow):**
   $$\text{CLV}_{\text{pred}} = \sum_{t=1}^{24} \frac{\text{Monthly Gross Profit} \times r^t}{(1 + d)^t}$$
   where $r$ represents the retention probability and $d$ represents the monthly cost of capital (10% annualized).
3. **LTV:CAC Ratio:** Compares total expected customer lifetime value against marketing acquisition costs across acquisition channels.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

from src.analytics.eda import load_curated_data
from src.clv.clv_model import calculate_customer_clv_metrics, clv_breakdown_by_channel

df_cust, df_prod, df_ord, df_items = load_curated_data()
print("Data loaded.")

## 1. Calculate Customer CLV Metrics

In [ ]:
df_clv = calculate_customer_clv_metrics(df_cust, df_ord, df_items)

print(f"Customer CLV Metrics computed for {len(df_clv):,} active customers.")
df_clv[['customer_id', 'acquisition_channel', 'historical_profit', 'predictive_24m_clv_profit', 'total_expected_clv', 'cac_bdt', 'clv_to_cac_ratio']].head(10)

## 2. Marketing Acquisition Channel Efficiency (LTV:CAC Benchmarking)

In [ ]:
channel_clv = clv_breakdown_by_channel(df_clv)
display(channel_clv)

plt.figure(figsize=(10, 5))
plt.bar(channel_clv['acquisition_channel'], channel_clv['ltv_cac_ratio'], color='#2ca02c', width=0.5)
plt.axhline(3.0, color='red', linestyle='--', label='Healthy Benchmark (3.0x)')
plt.title('LTV:CAC Ratio across Acquisition Channels in Bangladesh', fontsize=13, fontweight='bold')
plt.xlabel('Marketing Channel')
plt.ylabel('LTV : CAC Ratio')
plt.xticks(rotation=25, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## 3. CLV by Administrative Division

In [ ]:
div_clv = df_clv.groupby('division').agg(
    customer_count=('customer_id', 'count'),
    avg_total_clv=('total_expected_clv', 'mean'),
    median_clv=('total_expected_clv', 'median'),
    total_division_clv=('total_expected_clv', 'sum')
).reset_index().sort_values(by='total_division_clv', ascending=False)

display(div_clv)